In [ ]:
import os
import random
import argparse
from collections import deque
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
from torch.utils.data import DataLoader

from utils import CustomDataset, init_weights, compute_metrics
from utils import run_epoch_Classification_Network, run_epoch_Segmentation_Network, run_epoch_Dual_Task

from Model.DHD_Net.Classification_Network import Classification_Network
from Model.DHD_Net.Segmentation_Network import Segmentation_Network
from Model.DHD_Net.Dual_Task_Fusion_Network import Dual_Task

In [ ]:
# ------------------------ Arguments ------------------------
parser = argparse.ArgumentParser(description='End-to-end training pipeline')
parser.add_argument('--manual_seed', type=int, default=2000)
parser.add_argument('--class_num', type=int, default=6, help='Number of classes for classification')
parser.add_argument('--seg_class_num', type=int, default=2, help='Number of classes for segmentation')
parser.add_argument('--re_size', default=None, help='Resize size, e.g., 256')
parser.add_argument('--extension_img', type=str, default='tif')
parser.add_argument('--extension_lab', type=str, default='tif')
parser.add_argument('--device', type=str, default='cuda:1')

parser.add_argument('--train_img_dir', type=str, default='./Data/Dataset/train/data')
parser.add_argument('--train_lab_dir', type=str, default='./Data/Dataset/train/label')
parser.add_argument('--val_img_dir', type=str, default='./Data/Dataset/val/data')
parser.add_argument('--val_lab_dir', type=str, default='./Data/Dataset/val/label')

parser.add_argument('--final_model_path', type=str, default='./Model_Weight/DHD-Net.pkl')

parser.add_argument('--lr', type=float, default=0.001)
parser.add_argument('--beta1', type=float, default=0.9)
parser.add_argument('--beta2', type=float, default=0.999)
parser.add_argument('--weight_decay', type=float, default=0.01)

parser.add_argument('--patience', type=int, default=15)
parser.add_argument('--smooth_window', type=int, default=7)

parser.add_argument('--class_epochs', type=int, default=150)
parser.add_argument('--seg_epochs', type=int, default=150)
parser.add_argument('--dual_epochs', type=int, default=150)

parser.add_argument('--batch_size', type=int, default=8)

opt = parser.parse_args(args=[])

device = torch.device(opt.device if torch.cuda.is_available() and 'cuda' in opt.device else 'cpu')
print(opt, "\n", device)

random.seed(opt.manual_seed)
torch.manual_seed(opt.manual_seed)
cudnn.benchmark = True
print(f"Random Seed: {opt.manual_seed}")

In [ ]:
# ------------------------ Stage 1: Classification ------------------------
print("\n" + "="*60)
print("Stage 1: Training Classification Network (skip zero-label samples)")
print("="*60)

class_train_ds = CustomDataset(opt.train_img_dir, opt.train_lab_dir,
                               extension_img=opt.extension_img, extension_lab=opt.extension_lab,
                               image_size=opt.re_size, skip_zero_label=True)
class_val_ds = CustomDataset(opt.val_img_dir, opt.val_lab_dir,
                             extension_img=opt.extension_img, extension_lab=opt.extension_lab,
                             image_size=opt.re_size, skip_zero_label=True)

class_train_loader = DataLoader(class_train_ds, batch_size=opt.batch_size, shuffle=True)
class_val_loader = DataLoader(class_val_ds, batch_size=opt.batch_size, shuffle=False)

class_model = Classification_Network(opt.class_num).to(device)
class_optimizer = optim.AdamW(class_model.parameters(), lr=opt.lr,
                              betas=(opt.beta1, opt.beta2), weight_decay=opt.weight_decay)
class_criterion = nn.CrossEntropyLoss(ignore_index=0)
class_model.apply(init_weights)

best_class_score = -float('inf')
best_class_state = None
class_no_improve = 0
class_smoothed = deque(maxlen=opt.smooth_window)
best_class_smoothed = -float('inf')

for epoch in range(opt.class_epochs):
    train_loss, train_true, train_pred = run_epoch_Classification_Network(
        class_train_loader, class_model, class_criterion, class_optimizer, is_train=True, device=device
    )
    train_metrics = compute_metrics(train_true.cpu().numpy(), train_pred.cpu().numpy(), num_classes=opt.class_num)
    val_loss, val_true, val_pred = run_epoch_Classification_Network(
        class_val_loader, class_model, class_criterion, optimizer=None, is_train=False, device=device
    )
    val_metrics = compute_metrics(val_true.cpu().numpy(), val_pred.cpu().numpy(), num_classes=opt.class_num)

    class_smoothed.append(val_metrics["MRecall"])
    smoothed_val = np.mean(class_smoothed)

    saved = False
    if val_metrics["MRecall"] > best_class_score:
        best_class_score = val_metrics["MRecall"]
        best_class_state = {k: v.cpu().clone() for k, v in class_model.state_dict().items()}
        saved = True

    if smoothed_val > best_class_smoothed:
        best_class_smoothed = smoothed_val
        class_no_improve = 0
    else:
        class_no_improve += 1

    msg = (f"Epoch {epoch+1:3d}/{opt.class_epochs} | Train Loss: {train_loss:.4f} | "
           f"Train MRecall: {train_metrics['MRecall']:.4f} | Val Loss: {val_loss:.4f} | "
           f"Val MRecall: {val_metrics['MRecall']:.4f}")
    if saved:
        msg += f" | ★ Best: {best_class_score:.4f} (in memory)"
    msg += f" | Patience left: {opt.patience - class_no_improve}"

    if class_no_improve >= opt.patience:
        msg += " | Early stopping"
        print(msg)
        break
    print(msg)

if best_class_state is not None:
    class_model.load_state_dict(best_class_state)

del class_model, class_optimizer
torch.cuda.empty_cache()
print("Stage 1 finished, GPU memory released.\n")

In [ ]:
# ------------------------ Stage 2: Segmentation ------------------------
print("="*60)
print("Stage 2: Training Segmentation Network")
print("="*60)

seg_train_ds = CustomDataset(opt.train_img_dir, opt.train_lab_dir,
                             extension_img=opt.extension_img, extension_lab=opt.extension_lab,
                             image_size=opt.re_size, skip_zero_label=False)
seg_val_ds = CustomDataset(opt.val_img_dir, opt.val_lab_dir,
                           extension_img=opt.extension_img, extension_lab=opt.extension_lab,
                           image_size=opt.re_size, skip_zero_label=False)
seg_train_loader = DataLoader(seg_train_ds, batch_size=opt.batch_size, shuffle=True)
seg_val_loader = DataLoader(seg_val_ds, batch_size=opt.batch_size, shuffle=False)

seg_model = Segmentation_Network(opt.seg_class_num).to(device)
seg_optimizer = optim.AdamW(seg_model.parameters(), lr=opt.lr,
                            betas=(opt.beta1, opt.beta2), weight_decay=opt.weight_decay)
seg_criterion = nn.CrossEntropyLoss()
seg_model.apply(init_weights)

best_seg_score = -float('inf')
best_seg_state = None
seg_no_improve = 0
seg_smoothed = deque(maxlen=opt.smooth_window)
best_seg_smoothed = -float('inf')

for epoch in range(opt.seg_epochs):
    train_loss, train_true, train_pred = run_epoch_Segmentation_Network(
        seg_train_loader, seg_model, seg_criterion, seg_optimizer, is_train=True, device=device
    )
    train_metrics = compute_metrics(train_true.cpu().numpy(), train_pred.cpu().numpy(), num_classes=opt.seg_class_num)
    val_loss, val_true, val_pred = run_epoch_Segmentation_Network(
        seg_val_loader, seg_model, seg_criterion, optimizer=None, is_train=False, device=device
    )
    val_metrics = compute_metrics(val_true.cpu().numpy(), val_pred.cpu().numpy(), num_classes=opt.seg_class_num)

    seg_smoothed.append(val_metrics["Binary IoU"])
    smoothed_val = np.mean(seg_smoothed)

    saved = False
    if val_metrics["Binary IoU"] > best_seg_score:
        best_seg_score = val_metrics["Binary IoU"]
        best_seg_state = {k: v.cpu().clone() for k, v in seg_model.state_dict().items()}
        saved = True

    if smoothed_val > best_seg_smoothed:
        best_seg_smoothed = smoothed_val
        seg_no_improve = 0
    else:
        seg_no_improve += 1

    msg = (f"Epoch {epoch+1:3d}/{opt.seg_epochs} | Train Loss: {train_loss:.4f} | "
           f"Train Binary IoU: {train_metrics['Binary IoU']:.4f} | Val Loss: {val_loss:.4f} | "
           f"Val Binary IoU: {val_metrics['Binary IoU']:.4f}")
    if saved:
        msg += f" | ★ Best: {best_seg_score:.4f} (in memory)"
    msg += f" | Patience left: {opt.patience - seg_no_improve}"

    if seg_no_improve >= opt.patience:
        msg += " | Early stopping"
        print(msg)
        break
    print(msg)

if best_seg_state is not None:
    seg_model.load_state_dict(best_seg_state)

del seg_model, seg_optimizer
torch.cuda.empty_cache()
print("Stage 2 finished, GPU memory released.\n")

In [ ]:
# ------------------------ Stage 3: Dual-Task ------------------------
print("="*60)
print("Stage 3: Training Dual-Task Network (pretrained encoders frozen)")
print("="*60)

dual_train_ds = CustomDataset(opt.train_img_dir, opt.train_lab_dir,
                              extension_img=opt.extension_img, extension_lab=opt.extension_lab,
                              image_size=opt.re_size, skip_zero_label=False)
dual_val_ds = CustomDataset(opt.val_img_dir, opt.val_lab_dir,
                            extension_img=opt.extension_img, extension_lab=opt.extension_lab,
                            image_size=opt.re_size, skip_zero_label=False)
dual_train_loader = DataLoader(dual_train_ds, batch_size=opt.batch_size, shuffle=True)
dual_val_loader = DataLoader(dual_val_ds, batch_size=opt.batch_size, shuffle=False)

dual_model = Dual_Task(opt.class_num).to(device)
dual_optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, dual_model.parameters()),
    lr=opt.lr, betas=(opt.beta1, opt.beta2), weight_decay=opt.weight_decay
)
dual_criterion = nn.CrossEntropyLoss()
dual_model.apply(init_weights)

# Load and freeze classification encoder (stage_*)
for i in range(5):
    stage_name = f"stageMS_{i}"
    stage_weights = {k.replace(f"{stage_name}.", ""): v
                     for k, v in best_class_state.items() if k.startswith(stage_name)}
    getattr(dual_model, stage_name).load_state_dict(stage_weights)
    for param in getattr(dual_model, stage_name).parameters():
        param.requires_grad = False

# Load and freeze segmentation encoder (stageSAR_*)
for i in range(5):
    stage_name = f"stageSAR_{i}"
    stage_weights = {k.replace(f"{stage_name}.", ""): v
                     for k, v in best_seg_state.items() if k.startswith(stage_name)}
    getattr(dual_model, stage_name).load_state_dict(stage_weights)
    for param in getattr(dual_model, stage_name).parameters():
        param.requires_grad = False

print("Encoders loaded and frozen.")

best_dual_score = -float('inf')
best_dual_state = None
dual_no_improve = 0
dual_smoothed = deque(maxlen=opt.smooth_window)
best_dual_smoothed = -float('inf')

for epoch in range(opt.dual_epochs):
    train_loss, train_true, train_pred = run_epoch_Dual_Task(
        dual_train_loader, dual_model, dual_criterion, dual_optimizer, is_train=True, device=device
    )
    train_metrics = compute_metrics(train_true.cpu().numpy(), train_pred.cpu().numpy(), num_classes=opt.class_num)
    val_loss, val_true, val_pred = run_epoch_Dual_Task(
        dual_val_loader, dual_model, dual_criterion, optimizer=None, is_train=False, device=device
    )
    val_metrics = compute_metrics(val_true.cpu().numpy(), val_pred.cpu().numpy(), num_classes=opt.class_num)

    dual_smoothed.append(val_metrics["F1_1"])
    smoothed_val = np.mean(dual_smoothed)

    saved = False
    if val_metrics["F1_1"] > best_dual_score:
        best_dual_score = val_metrics["F1_1"]
        best_dual_state = {k: v.cpu().clone() for k, v in dual_model.state_dict().items()}
        saved = True

    if smoothed_val > best_dual_smoothed:
        best_dual_smoothed = smoothed_val
        dual_no_improve = 0
    else:
        dual_no_improve += 1

    msg = (f"Epoch {epoch+1:3d}/{opt.dual_epochs} | Train Loss: {train_loss:.4f} | "
           f"Train F1_1: {train_metrics['F1_1']:.4f} | Val Loss: {val_loss:.4f} | "
           f"Val F1_1: {val_metrics['F1_1']:.4f}")
    if saved:
        msg += f" | ★ Best: {best_dual_score:.4f} (saved to file)"
        torch.save(best_dual_state, opt.final_model_path)
    msg += f" | Patience left: {opt.patience - dual_no_improve}"

    if dual_no_improve >= opt.patience:
        msg += " | Early stopping"
        print(msg)
        break
    print(msg)

if best_dual_state is not None and (not os.path.exists(opt.final_model_path) or best_dual_score > 0):
    torch.save(best_dual_state, opt.final_model_path)
    print(f"Final model saved to {opt.final_model_path} (best F1_1: {best_dual_score:.4f})")

print("\n" + "="*60)
print("End-to-end training completed.")
print("="*60)